In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from astropy.visualization import quantity_support
quantity_support()
# from scipy.stats._stats_py import Power_divergenceResult
%matplotlib inline
from spectrum_component_analyser.readers.JWST.instruments import NIRISS, Instrument, NIRSPECLower
from spectrum_component_analyser.readers.JWST.folder_reader import JWSTFolderReader
from spectrum_component_analyser.readers.JWST.target import *
from spectrum_component_analyser.spectrum import spectrum

target : JWSTTarget = TRAPPIST1
instrument : Instrument = NIRISS

all_spectra : list[spectrum] = JWSTFolderReader.get_all_spectra(target=target, instrument=instrument)

# print(all_spectra)

flux_matrix = np.array([spec.Fluxes for spec in all_spectra])

# 2. Create a mask: True only if ALL spectra at that wavelength are finite and > 0
# axis=0 looks across the different spectra for each wavelength point
mask = np.all(np.isfinite(flux_matrix) & (flux_matrix > 0) & (np.abs(flux_matrix) < 1e9), axis=0)

example_spectrum = all_spectra[0]

# mask = np.all(np.isfinite(all_spectra.Fluxes)) & np.all(all_spectra.Fluxes > 0)

s : spectrum
for s in all_spectra:
    s[mask].plot(clear=False)
plt.show()

In [ ]:
# draw light curve

indices = []
fluxes = []

s : spectrum
for index, s in enumerate(all_spectra):
    indices.append(index)
    fluxes.append(np.sum(s.Fluxes[mask].value))

fig, ax = plt.subplots(figsize=(8,8))
ax.plot(indices, fluxes)
plt.show()

for s in all_spectra:
    s.normalise_flux()

In [ ]:
"""
find the standard deviation of each point
find an averaged spectrum
analyse that average spectrum using a (now correct) Pearson chi-squared test
"""

min_included_integration_index = 0
max_included_integration_index = 80

all_fluxes=[spec.Fluxes for spec in all_spectra[min_included_integration_index:max_included_integration_index]]

average_spectrum = spectrum(wavelengths=all_spectra[0].Wavelengths,
                                 fluxes=np.mean(all_fluxes, axis=0) * all_fluxes[0].unit,
                                 normalised_point=None,
                                 temperature = None,
                                 observational_wavelengths=None,
                                 name="averaged " + target.name)

average_spectrum = average_spectrum[mask]

flux_std_deviation = np.std(all_fluxes, axis=0) * all_fluxes[0].unit

flux_std_deviation = flux_std_deviation[mask]

fig, ax = plt.subplots(figsize=(26,10))
average_spectrum.plot()
plt.errorbar(average_spectrum.Wavelengths, average_spectrum.Fluxes, yerr=flux_std_deviation, ecolor='red', capsize=2)
plt.scatter(average_spectrum.Wavelengths, flux_std_deviation, s=1)
plt.show()


In [ ]:
plt.clf()

from constants import *
from astropy import units as u

from phoenix_grid_creator.spectral_grid import spectral_grid

fits_file_paths = list(Path(package_path / "raw_phoenix_spectra").rglob("*.fits"))
# fits_file_paths =fits_file_paths[:1]
spec_grid : spectral_grid = spectral_grid.from_local_raw(
    fits_file_paths,
    instrument.Resolution,
    False,
    observational_wavelengths=average_spectrum.Wavelengths) # need to downsample resolution otherwise the spectral grid won't fit in memory...

# units are getting lost somewhere, idk why
spec_grid.Wavelengths = spec_grid.Wavelengths.to(u.um)
spec_grid.Fluxes *= u.Jy

spec_grid.get_spectrum(spec_grid.T_effs[-1], spec_grid.FeHs[0], spec_grid.Log_gs[0]).plot()

In [ ]:
import numpy as np
from spectrum_component_analyser.chi_squared_minimisation import ChiHelper

number_of_parameters : int = 4 # weight, t, f, l

%matplotlib inline
number_of_components : int = 1

chi : ChiHelper = ChiHelper(
    spec_grid=spec_grid,
    number_of_components=number_of_components,
    number_of_parameters=number_of_parameters,
    observed_spectrum=average_spectrum
)

parameter_bounds = [
        (.0, 2.),
        (np.min(spec_grid.T_effs.value), np.max(spec_grid.T_effs.value)),
        (np.min(spec_grid.FeHs.value), np.max(spec_grid.FeHs.value)),
        # (target.feh.value - 0.01, target.feh.value + 0.01),
        # (0, 0),
        (np.min(spec_grid.Log_gs.value), np.max(spec_grid.Log_gs.value)),
        # (target.log_g.value - 0.01, target.log_g.value + 0.01)
        # (1.9, 2.1),
    ]

r = chi.get_r(parameter_bounds)

print(r)

In [ ]:
%matplotlib inline

print("simulated / true")

chi.plot(r)

In [ ]:
# unconstrained optimisation
# from spectrum_component_analyser.mcmc.helper import MCMCHelper

# mcmc : MCMCHelper = MCMCHelper(
#     parameter_bounds=parameter_bounds,
#     number_of_components=number_of_components,
#     number_of_parameters=number_of_parameters,
#     observed_spectrum=average_spectrum,
#     spec_grid=spec_grid,
#     n_steps = 2000,
#     n_walkers=64
# )

# sampler, samples = mcmc.run(r)


In [ ]:
# optimisation is constrained so that feh and log g are the same in all components

from spectrum_component_analyser.mcmc.constrained_helper import ConstrainedMCMCHelper

mcmc = ConstrainedMCMCHelper(
    parameter_bounds=parameter_bounds,  # Still [(w_min, w_max), (t_min, t_max), (f_min, f_max), (l_min, l_max)]
    number_of_components=number_of_components,
    observed_spectrum=average_spectrum,
    spec_grid=spec_grid,
    n_steps=2000,
    n_walkers=64
)

# r.x can still be the "flat" 4N array from your global optimizer.
# The class will automatically compress it to the shared FeH/logg format.
sampler, samples = mcmc.run(r)

In [ ]:
%matplotlib inline

mcmc.plot_corner(sampler, samples, true_components=None)

In [ ]:
mcmc.print_parameters(samples=samples)

In [ ]:
%matplotlib widget
mcmc.plot_spectrum(samples)